<a href="https://colab.research.google.com/github/BWMIN-Hub/SR_practice/blob/main/notebooks/20_misr_vs_sisr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MISR vs SISR

| | input | output |
|---|---|---|
| SISR | one LR image | HR x3 |
| MISR | several LR images of the same place | HR x3 |

Each LR image is degraded independently, so each one keeps and loses slightly different
detail.  MISR can combine them; SISR has only one.

The data: 64 validation patches, each with nine 64 px LR images and the 192 px HR.

| model | network |
|---|---|
| SISR | EDSR, takes LR image 1 |
| MISR | 3D UNet, takes nine LR images |

## 1. Setup

In [ ]:
import sys, urllib.request

BASE = 'https://raw.githubusercontent.com/BWMIN-Hub/SR_practice/main'
for f in ('sr_models.py', 'misr_models.py', 'iqa.py'):
    urllib.request.urlretrieve(f'{BASE}/lib/{f}', f)
    sys.modules.pop(f[:-3], None)

import cv2, numpy as np, torch
import matplotlib.pyplot as plt
from sr_models import load_edsr
from misr_models import load_misr
from iqa import psnr, ssim

get = lambda p: urllib.request.urlretrieve(f'{BASE}/{p}', p.split('/')[-1])[0]
z = np.load(get('dataset/misr/val.npz'))
LR, HR = z['lr'], z['hr']                       # (64, 9, 64, 64, 3), (64, 192, 192, 3)
CK = 'models/20_misr_vs_sisr/checkpoints'
sisr = load_edsr(get(f'{CK}/edsr_sisr_x3.pt')).eval()
misr = load_misr(get(f'{CK}/misr_x3_t9.pt'))
dev = next(misr.parameters()).device


def run_sisr(lr):
    """lr: (N, h, w, 3) uint8 -> (N, 3h, 3w, 3) uint8"""
    x = torch.from_numpy(lr.astype(np.float32)).permute(0, 3, 1, 2).to(dev)
    with torch.no_grad():
        y = torch.cat([sisr(b) for b in x.split(16)]).clamp(0, 255)
    return y.round().byte().permute(0, 2, 3, 1).cpu().numpy()


def run_misr(lr, k=9):
    """lr: (N, 9, h, w, 3) uint8.  The first k images fill the nine slots in turn."""
    lr = lr[:, [i % k for i in range(9)]]
    x = torch.from_numpy(lr.astype(np.float32) / 127.5 - 1).permute(0, 4, 1, 2, 3).to(dev)
    with torch.no_grad():
        y = torch.cat([misr(b) for b in x.split(16)]).clamp(-1, 1)
    return ((y + 1) * 127.5).round().byte().permute(0, 2, 3, 1).cpu().numpy()


def bicubic(lr):
    return np.stack([cv2.resize(a, (a.shape[1] * 3, a.shape[0] * 3),
                                interpolation=cv2.INTER_CUBIC) for a in lr])


def scores(pred):
    return (np.mean([psnr(p, h) for p, h in zip(pred, HR)]),
            np.mean([ssim(p, h) for p, h in zip(pred, HR)]))

## 2. The inputs

Nine LR images of one patch.  The bottom row is each image minus the mean of the nine,
stretched: where they differ.

In [ ]:
i = 26
mean = LR[i].astype(np.float32).mean(0)
D = (LR[i].astype(np.float32) - mean).mean(3)
v = np.percentile(np.abs(D), 99)
fig, ax = plt.subplots(2, 10, figsize=(20, 4.4))
for j in range(9):
    ax[0][j].imshow(LR[i, j], interpolation='nearest'); ax[0][j].set_title(f'LR {j + 1}')
    ax[1][j].imshow(D[j], cmap='RdBu', vmin=-v, vmax=v, interpolation='nearest')
ax[0][9].imshow(HR[i]); ax[0][9].set_title('HR')
ax[1][9].axis('off')
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

## 3. SISR vs MISR

Same patch, same HR.  PSNR and SSIM against the HR are in the titles.

In [ ]:
def compare(i):
    panels = [('LR 1', cv2.resize(LR[i, 0], (192, 192), interpolation=cv2.INTER_NEAREST)),
              ('Bicubic', bicubic(LR[i:i + 1, 0])[0]),
              ('SISR', run_sisr(LR[i:i + 1, 0])[0]),
              ('MISR (9)', run_misr(LR[i:i + 1])[0]),
              ('HR', HR[i])]
    fig, ax = plt.subplots(1, 5, figsize=(20, 4.6))
    for a, (name, im) in zip(ax, panels):
        a.imshow(im)
        a.set_title(name if name in ('LR 1', 'HR') else
                    f'{name}\n{psnr(im, HR[i]):.2f} dB  SSIM {ssim(im, HR[i]):.3f}')
        a.set_xticks([]); a.set_yticks([])
    plt.tight_layout(); plt.show()


compare(26)
compare(63)

## 4. Number of inputs

MISR with the first k LR images, k = 1 - 9, averaged over all 64 patches.  The network
always has nine slots; with k images they fill the slots in turn (k = 2: 1 2 1 2 ...).
SISR and bicubic use LR 1 only, so they are flat lines.

In [ ]:
K = range(1, 10)
m = np.array([scores(run_misr(LR, k)) for k in K])
s = scores(run_sisr(LR[:, 0]))
b = scores(bicubic(LR[:, 0]))

fig, ax = plt.subplots(1, 2, figsize=(13, 4.4))
for j, name in enumerate(('PSNR (dB)', 'SSIM')):
    ax[j].plot(K, m[:, j], 'o-', color='#2b6cb0', lw=2, label='MISR')
    ax[j].axhline(s[j], color='#c0504d', lw=2, ls='--', label='SISR')
    ax[j].axhline(b[j], color='#888888', lw=1.5, ls=':', label='bicubic')
    ax[j].set_xlabel('number of LR inputs'); ax[j].set_title(name, loc='left')
    ax[j].set_xticks(list(K)); ax[j].grid(alpha=.3)
ax[0].legend(frameon=False)
plt.tight_layout(); plt.show()

print(f'{"":10s} {"PSNR":>7s} {"SSIM":>7s}')
print(f'{"bicubic":10s} {b[0]:7.2f} {b[1]:7.4f}')
print(f'{"SISR":10s} {s[0]:7.2f} {s[1]:7.4f}')
for k, (p, q) in zip(K, m):
    print(f'{f"MISR k={k}":10s} {p:7.2f} {q:7.4f}')

PSNR and SSIM rise with every added image.  The second image adds 1.9 dB, the third
1.0 dB, the fourth 0.5 dB, and each one after that 0.15 - 0.3 dB.

MISR passes SISR at k = 2.  At k = 1 it is below SISR — it was trained with nine
different images, and nine copies of one image is not what it learned.

## 5. What the extra inputs add

The same patch with k = 1, 3, 9.

In [ ]:
i = 63
fig, ax = plt.subplots(1, 4, figsize=(16, 4.6))
for a, k in zip(ax, (1, 3, 9)):
    im = run_misr(LR[i:i + 1], k)[0]
    a.imshow(im); a.set_title(f'MISR k={k}\n{psnr(im, HR[i]):.2f} dB  SSIM {ssim(im, HR[i]):.3f}')
ax[3].imshow(HR[i]); ax[3].set_title('HR')
for a in ax:
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()